# P001 0-1 背包问题 · PySCIPOpt 建模与评测演示

**场景**：在有限预算或载荷容量上限下，从若干不可分割的候选物料或项目中遴选组合，使总收益最大。本案例在浏览器（WebAssembly + Python）中直接运行 `from pyscipopt import Model, quicksum`，无需本地配置环境。

练习用，不计正式成绩。

## 1. 混合整数线性规划模型

- 二元变量 $x_i \in \{0, 1\}$ 表示第 $i$ 件物品是否选入背包；
- 目标函数：最大化所选物品总价值：

$$\max \sum_{i=0}^{n-1} v_i x_i$$

- 容量约束：选中物品总重量不超过容量上限 $\sum_{i=0}^{n-1} w_i x_i \le C$。

In [ ]:
from pyscipopt import Model, quicksum
import numpy as np
print("PySCIPOpt 与 NumPy 在浏览器 WebAssembly 环境中加载成功！")

## 2. 接口契约：solve(instance) -> dict

输入数据规范：
- `n`：物品数量；
- `weights`：物品重量列表 `[w0, w1, ...]`；
- `values`：物品价值列表 `[v0, v1, ...]`；
- `capacity`：背包容量上限。

返回值规范：
- `status`：求解状态（如 `"optimal"`）；
- `objective`：`float` 目标值（选入物品的总价值）；
- `solution`：物品下标映射 `dict`（`0..n-1 -> 0.0 / 1.0`）或长度为 `n` 的列表。

In [ ]:
def solve(instance: dict) -> dict:
    weights = instance["weights"]
    values = instance["values"]
    capacity = instance["capacity"]
    n = instance["n"]

    model = Model("knapsack")
    model.hideOutput(True)

    # 0-1 变量：x_i 表示第 i 个物品是否装入背包
    x = [model.addVar(vtype="B", name=f"x_{i}") for i in range(n)]

    # 目标函数：最大化总价值
    model.setObjective(quicksum(values[i] * x[i] for i in range(n)), "maximize")

    # 容量约束
    model.addCons(quicksum(weights[i] * x[i] for i in range(n)) <= capacity)

    model.optimize()

    sol = {i: float(round(model.getVal(x[i]))) for i in range(n)}
    return {
        "status": "optimal" if model.getStatus() == "optimal" else str(model.getStatus()),
        "objective": float(model.getObjVal()),
        "solution": sol,
    }

In [ ]:
# 测试用例验证
test_instance = {
    "n": 4,
    "weights": [2, 3, 4, 5],
    "values": [3, 4, 5, 6],
    "capacity": 8
}
res = solve(test_instance)
print("求解结果:", res)

## 3. 可以接着改什么

1. 将 `capacity` 从 `8` 调整为 `7` 或 `10`，观察选中物品组合与单位重量价值排序规律；
2. 把某件物品价值调低后重跑，检验「性价比高但占容量大」的物品何时被多个小物品替代；
3. 正式提交 `P001` 作业前，请前往「课程教学 → 工作台」用公开样例点击 **「本地自测」** 校验。